[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/solutions/exercises/19_imbalanced_learning/90_exercises.ipynb)

# Bài tập — Chương 19: Imbalanced Learning (L0 → L7)

> 📗 **BẢN LỜI GIẢI** — chỉ mở sau khi đã tự làm bài. Bản bài tập (đã xoá lời giải) nằm trong thư mục `notebooks/` của chương.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/solutions/exercises/19_imbalanced_learning

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn import metrics as skm
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_validate, train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from ml_course.checks import check_close, check_equal, check_greater, check_less, check_true
from ml_course.utils import repo_path

rng = np.random.default_rng(0)

## L0 — Nhớ khái niệm

**Câu 1.** Nêu: 3 metric phù hợp cho dữ liệu mất cân bằng; công thức trọng số `class_weight="balanced"`; ý tưởng SMOTE; công thức hiệu chỉnh prior sau resampling.

✍️ Trả lời:

**Đáp án:** PR-AUC (average precision), precision/recall (hoặc $F_\beta$) tại ngưỡng vận hành, precision@k / recall@k, chi phí kỳ vọng, MCC. Balanced: $w_c = \frac{n}{K\,n_c}$. SMOTE: mẫu mới $= \mathbf x_i + u(\mathbf x_{nn} - \mathbf x_i)$, $u \sim U(0, 1)$, với $\mathbf x_{nn}$ là một trong $k$ láng giềng thiểu số gần nhất. Hiệu chỉnh prior: $p = \frac{p_s\pi/\pi_s}{p_s\pi/\pi_s + (1 - p_s)(1 - \pi)/(1 - \pi_s)}$.

## L1 — Giải thích bằng lời

**Câu 2.** Một đồng nghiệp nói: "Dữ liệu chỉ 1% gian lận nên model chẳng học được gì — ta phải SMOTE về 50/50 thì mới dùng được". Phản biện có căn cứ.

✍️ Trả lời:

**Gợi ý:** model có thể đã xếp hạng tốt (PR-AUC cao hơn nhiều so với 0.01) — vấn đề là **ngưỡng 0.5** và metric accuracy. SMOTE về 50/50 thường không cải thiện khả năng xếp hạng (ROC/PR-AUC), chỉ dịch xác suất lên (tương đương hạ ngưỡng) và làm hỏng calibration; còn có thể tạo mẫu sai ở vùng đan xen. Nên: dùng PR-AUC/chi phí, chọn ngưỡng theo chi phí trên validation, thử class weight; nếu vẫn kém thì vấn đề là **số lượng** mẫu gian lận hoặc feature — thu thập thêm nhãn, feature engineering.

## L2 — Tính tay

**Câu 3.** (a) 10 000 mẫu, 200 positive: trọng số `balanced` của mỗi lớp? (b) Model train trên dữ liệu undersample 50/50 dự đoán $p_s = 0.6$; tỉ lệ positive thật $\pi = 0.02$. Xác suất đã hiệu chỉnh?

In [ ]:
### START CODE HERE ### (≈ 5 dòng)
w0 = 10_000 / (2 * 9_800)
w1 = 10_000 / (2 * 200)
a = 0.6 * 0.02 / 0.5
b = 0.4 * 0.98 / 0.5
p_corrected = a / (a + b)
### END CODE HERE ###

In [ ]:
check_close((w0, w1), (10_000 / 19_600, 25.0), message="trọng số balanced")
check_close(p_corrected, 0.012 / (0.012 + 0.392), tol=1e-9, message="xác suất hiệu chỉnh ≈ 0.0297")

**Câu 4.** Chi phí: FP = 1, FN = 50. Với xác suất **calibrated**, ngưỡng tối ưu? Nếu model đã train với `class_weight` nhân trọng số lớp positive lên 50 lần, ngưỡng tương ứng xấp xỉ là bao nhiêu (gợi ý: weight nhân odds lên 50)?

In [ ]:
### START CODE HERE ### (≈ 3 dòng)
t_calibrated = 1 / (1 + 50)
odds_t = t_calibrated / (1 - t_calibrated) * 50
t_weighted = odds_t / (1 + odds_t)
### END CODE HERE ###

In [ ]:
check_close(t_calibrated, 1 / 51, message="ngưỡng với xác suất calibrated")
check_close(t_weighted, 0.5, message="weight 50 = dịch ngưỡng: ngưỡng 0.5 trên model có trọng số")

## L3 — Code

**Bài 5.** Cài đặt `smote(X, y, k, seed)` (cân bằng hoàn toàn hai lớp) — dùng `NearestNeighbors` trên **lớp thiểu số**. Kiểm tra: số mẫu mỗi lớp bằng nhau; mọi mẫu mới nằm trên đoạn thẳng giữa hai mẫu thiểu số gốc.

In [ ]:
def smote(X, y, k=5, seed=0):
    r = np.random.default_rng(seed)
    X_min = X[y == 1]
    n_new = int((y == 0).sum() - (y == 1).sum())
    ### START CODE HERE ### (≈ 6 dòng)
    # STARTER: neigh = None      # chỉ số k+1 láng giềng gần nhất trong X_min (cột 0 là chính nó) — NearestNeighbors(...).kneighbors(X_min)
    # STARTER: base = None       # n_new chỉ số mẫu gốc ngẫu nhiên
    # STARTER: partner = None    # với mỗi mẫu gốc, một láng giềng ngẫu nhiên (cột 1..k)
    # STARTER: u = None          # (n_new, 1) số ngẫu nhiên trong [0, 1)
    # STARTER: X_new = None      # X_min[base] + u * (X_min[partner] − X_min[base])
    _, neigh = NearestNeighbors(n_neighbors=k + 1).fit(X_min).kneighbors(X_min)
    base = r.integers(0, len(X_min), n_new)
    partner = neigh[base, r.integers(1, k + 1, n_new)]
    u = r.random((n_new, 1))
    X_new = X_min[base] + u * (X_min[partner] - X_min[base])
    ### END CODE HERE ###
    return np.vstack([X, X_new]), np.r_[y, np.ones(n_new, dtype=y.dtype)], (base, partner, X_new)

In [ ]:
X5 = np.vstack([rng.normal(0, 1, (300, 3)), rng.normal(1.0, 1.0, (30, 3))])
y5 = np.r_[np.zeros(300, int), np.ones(30, int)]
Xs, ys, (base, partner, X_new) = smote(X5, y5, k=5)
X_min = X5[y5 == 1]
check_equal(int((ys == 0).sum()), int((ys == 1).sum()), "hai lớp cân bằng")
# điểm mới nằm trên đoạn [x_base, x_partner]: x_new − x_base song song với x_partner − x_base, hệ số trong [0, 1]
d1, d2 = X_new - X_min[base], X_min[partner] - X_min[base]
t = (d1 * d2).sum(1) / np.maximum((d2 * d2).sum(1), 1e-12)
check_true(np.allclose(d1, t[:, None] * d2, atol=1e-9) and np.all((t >= -1e-9) & (t <= 1 + 1e-9)), "mẫu mới nằm trên đoạn thẳng")

**Bài 6.** Viết meta-estimator `ResampledClassifier(estimator, sampler)` (resample **chỉ trong `fit`**) để dùng được với `cross_val_score`. Lưu ý thứ tự kế thừa `ClassifierMixin, BaseEstimator` (sklearn ≥ 1.6).

In [ ]:
class ResampledClassifier(ClassifierMixin, BaseEstimator):
    def __init__(self, estimator, sampler=None):
        self.estimator, self.sampler = estimator, sampler

    def fit(self, X, y):
        ### START CODE HERE ### (≈ 5 dòng)
        # STARTER: # nếu có sampler: X, y = self.sampler(X, y)[:2]  (chỉ resample dữ liệu train trong fit)
        # STARTER: # self.estimator_ = clone(self.estimator).fit(X, y); self.classes_ = self.estimator_.classes_
        X, y = np.asarray(X, float), np.asarray(y)
        if self.sampler is not None:
            X, y = self.sampler(X, y)[:2]
        self.estimator_ = clone(self.estimator).fit(X, y)
        self.classes_ = self.estimator_.classes_
        ### END CODE HERE ###
        return self

    def predict_proba(self, X):
        return self.estimator_.predict_proba(np.asarray(X, float))

    def predict(self, X):
        return self.estimator_.predict(np.asarray(X, float))

In [ ]:
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
auc_plain = cross_val_score(ResampledClassifier(lr), X5, y5, cv=StratifiedKFold(5, shuffle=True, random_state=0), scoring="roc_auc").mean()
auc_smote = cross_val_score(ResampledClassifier(lr, smote), X5, y5, cv=StratifiedKFold(5, shuffle=True, random_state=0), scoring="roc_auc").mean()
print(f"ROC-AUC: không resample = {auc_plain:.4f} | SMOTE trong từng fold = {auc_smote:.4f}")
check_true(np.isfinite(auc_plain) and np.isfinite(auc_smote), "cross_val_score chạy được với meta-estimator")

## L4 — Debug

**Bài 7.** Quy trình sau có **3 lỗi**: (1) SMOTE trên toàn bộ dữ liệu **trước** khi chia; (2) báo cáo accuracy; (3) dùng ngưỡng 0.5 cho model train trên dữ liệu gốc (1% positive). Sửa: chia trước (stratified), resample (nếu muốn) **chỉ** trên train, báo cáo PR-AUC (`test_pr_auc`) và recall tại ngưỡng chọn trên validation để đạt precision ≥ 0.2 (`test_recall`).

In [ ]:
fraud = pd.read_csv(repo_path("data", "sample", "fraud_transactions.csv"))
feats = ["amount_vnd", "hour", "distance_from_home_km", "is_foreign", "n_tx_last_24h", "card_age_days"]
Xf = fraud[feats].astype(float).assign(amount_vnd=lambda d: np.log(d.amount_vnd)).to_numpy()
yf = fraud["is_fraud"].to_numpy()
### START CODE HERE ### (≈ 12 dòng)
# STARTER: Xs_all, ys_all, _ = smote(Xf, yf)                                                    # 🐛 (1)
# STARTER: X_tr, X_te, y_tr, y_te = train_test_split(Xs_all, ys_all, test_size=0.3, random_state=0)
# STARTER: m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(X_tr, y_tr)
# STARTER: p_te = m.predict_proba(X_te)[:, 1]
# STARTER: test_pr_auc = m.score(X_te, y_te)                                                    # 🐛 (2)
# STARTER: t_sel = 0.5                                                                          # 🐛 (3)
# STARTER: test_recall = skm.recall_score(y_te, p_te >= t_sel)
X_tmp, X_te, y_tmp, y_te = train_test_split(Xf, yf, test_size=0.3, random_state=0, stratify=yf)
X_tr, X_va, y_tr, y_va = train_test_split(X_tmp, y_tmp, test_size=0.3, random_state=0, stratify=y_tmp)
m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(X_tr, y_tr)
p_va, p_te = m.predict_proba(X_va)[:, 1], m.predict_proba(X_te)[:, 1]
prec, rec, thr = skm.precision_recall_curve(y_va, p_va)
ok = np.flatnonzero(prec[:-1] >= 0.2)
t_sel = thr[ok[0]]                                         # ngưỡng thấp nhất (recall cao nhất) vẫn đạt precision ≥ 0.2
test_pr_auc = skm.average_precision_score(y_te, p_te)
test_recall = skm.recall_score(y_te, p_te >= t_sel)
### END CODE HERE ###
print(f"ngưỡng = {t_sel:.4f} | PR-AUC test = {test_pr_auc:.3f} | recall test = {test_recall:.3f} | "
      f"precision test = {skm.precision_score(y_te, p_te >= t_sel, zero_division=0):.3f}")

In [ ]:
check_greater(test_pr_auc, 5 * y_te.mean(), "PR-AUC gấp ≥ 5 lần ngẫu nhiên")
check_greater(test_recall, 0.1, "recall tại ngưỡng đã chọn > 0.1 (ngưỡng 0.5 cho gần 0)")

## L5 — Phân tích

**Bài 8.** So sánh trên `fraud_transactions.csv` (5-fold stratified CV, Logistic Regression): không làm gì, `class_weight="balanced"`, random undersampling 1:1, SMOTE. Ghi ROC-AUC, PR-AUC, log-loss, và **Brier score** vào DataFrame `cmp`. Metric nào thay đổi nhiều, metric nào gần như không? Kết luận gì?

In [ ]:
def undersample(X, y, seed=0):
    r = np.random.default_rng(seed)
    pos, neg = np.flatnonzero(y == 1), np.flatnonzero(y == 0)
    keep = np.r_[pos, r.choice(neg, len(pos), replace=False)]
    return X[keep], y[keep]


cv8 = StratifiedKFold(5, shuffle=True, random_state=0)
scoring = {"roc_auc": "roc_auc", "pr_auc": "average_precision", "log_loss": "neg_log_loss", "brier": "neg_brier_score"}
### START CODE HERE ### (≈ 8 dòng)
# STARTER: variants = None   # dict tên → ResampledClassifier(...) cho 4 cách
# STARTER: rows = None       # với mỗi cách: cross_validate(..., scoring=scoring) → ROC-AUC, PR-AUC, log-loss, Brier (đổi dấu neg_*)
# STARTER: cmp = None        # DataFrame(rows).T
variants = {"không làm gì": ResampledClassifier(lr),
            "class_weight=balanced": ResampledClassifier(make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=2000))),
            "undersample 1:1": ResampledClassifier(lr, undersample),
            "SMOTE": ResampledClassifier(lr, smote)}
rows = {}
for name, est in variants.items():
    res = cross_validate(est, Xf, yf, cv=cv8, scoring=scoring)
    rows[name] = {"ROC-AUC": res["test_roc_auc"].mean(), "PR-AUC": res["test_pr_auc"].mean(),
                  "log-loss": -res["test_log_loss"].mean(), "Brier": -res["test_brier"].mean()}
cmp = pd.DataFrame(rows).T
### END CODE HERE ###
cmp.round(4)

In [ ]:
check_less(cmp["ROC-AUC"].max() - cmp["ROC-AUC"].min(), 0.03, "ROC-AUC gần như không đổi")
check_greater(cmp.loc["SMOTE", "log-loss"], 3 * cmp.loc["không làm gì", "log-loss"], "SMOTE làm hỏng xác suất (log-loss tăng mạnh)")

✍️ Kết luận:

**Gợi ý:** các kỹ thuật cân bằng gần như **không đổi** ROC-AUC/PR-AUC (khả năng xếp hạng) nhưng làm **log-loss và Brier tệ đi nhiều lần** — xác suất bị thổi phồng vì model tin tỉ lệ positive là 50%. Kết luận: với model xác suất tốt như Logistic Regression, "cân bằng dữ liệu" chủ yếu là cách hạ ngưỡng gián tiếp; chọn ngưỡng theo chi phí trên model gốc thường tốt hơn và giữ được xác suất có ý nghĩa.

## L6 — Thực tế

**Bài 9.** Đội kiểm soát rủi ro xử lý được **150 cảnh báo** mỗi ngày (giả sử tập test là "một ngày" giao dịch). Trên `fraud_transactions.csv` (tách test 30%, stratified, `random_state=1`): so sánh Logistic Regression và HistGradientBoosting (có và không `class_weight="balanced"`) theo **số gian lận bắt được trong top-150** (`caught`, dict tên → số). Chọn model theo đó. Yêu cầu model tốt nhất bắt ≥ 3 lần số gian lận của chọn ngẫu nhiên 150 giao dịch.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

Xd = fraud[feats].astype(float).assign(amount_vnd=lambda d: np.log(d.amount_vnd))
Xd = Xd.join(pd.get_dummies(fraud.merchant_category, prefix="cat", dtype=float)).to_numpy()
X9_tr, X9_te, y9_tr, y9_te = train_test_split(Xd, yf, test_size=0.3, random_state=1, stratify=yf)
### START CODE HERE ### (≈ 9 dòng)
# STARTER: models9 = None    # dict tên → 4 model (LR, LR balanced, HistGB, HistGB balanced)
# STARTER: caught = None     # dict tên → số gian lận trong 150 giao dịch có xác suất cao nhất của tập test
models9 = {"LR": make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)),
           "LR balanced": make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=3000)),
           "HistGB": HistGradientBoostingClassifier(learning_rate=0.05, max_leaf_nodes=15, random_state=0),
           "HistGB balanced": HistGradientBoostingClassifier(learning_rate=0.05, max_leaf_nodes=15, class_weight="balanced", random_state=0)}
caught = {}
for name, m9 in models9.items():
    p9 = m9.fit(X9_tr, y9_tr).predict_proba(X9_te)[:, 1]
    caught[name] = int(y9_te[np.argsort(-p9)[:150]].sum())
### END CODE HERE ###
random_expect = 150 * y9_te.mean()
print(caught, f"| chọn ngẫu nhiên kỳ vọng ≈ {random_expect:.1f}")

In [ ]:
check_greater(max(caught.values()), 3 * random_expect, "model tốt nhất bắt ≥ 3× ngẫu nhiên trong top-150")

## L7 — Nâng cao

**Bài 10.** **Ngưỡng phụ thuộc mẫu (example-dependent cost):** chi phí kiểm tra một cảnh báo = 30 000 VND; bỏ sót gian lận mất **toàn bộ** số tiền giao dịch $a_i$. Với xác suất calibrated $p_i$, nên cảnh báo khi kỳ vọng thiệt hại khi không cảnh báo lớn hơn chi phí kiểm tra: $p_i\,a_i > 30\,000$. So sánh tổng chi phí trên test (Bài 9, model LR không trọng số) của: (a) ngưỡng chung tối ưu trên validation; (b) quy tắc $p_i a_i > 30\,000$ (không cần chọn gì). Lưu `cost_global`, `cost_example`.

In [ ]:
amount = fraud["amount_vnd"].to_numpy()
X10_tmp, X10_te, y10_tmp, y10_te, a10_tmp, a10_te = train_test_split(Xd, yf, amount, test_size=0.3, random_state=1, stratify=yf)
X10_tr, X10_va, y10_tr, y10_va, a10_tr, a10_va = train_test_split(X10_tmp, y10_tmp, a10_tmp, test_size=0.3, random_state=0, stratify=y10_tmp)


def total_cost(y_true, alert, amounts):
    return (30_000 * np.sum(alert & (y_true == 0)) + np.sum(amounts[(~alert) & (y_true == 1)])) / 1e6


m10 = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)).fit(X10_tr, y10_tr)
p_va10, p_te10 = m10.predict_proba(X10_va)[:, 1], m10.predict_proba(X10_te)[:, 1]
### START CODE HERE ### (≈ 5 dòng)
ts = np.unique(np.quantile(p_va10, np.linspace(0.5, 0.9999, 400)))
t_best = ts[int(np.argmin([total_cost(y10_va, p_va10 >= t, a10_va) for t in ts]))]
cost_global = total_cost(y10_te, p_te10 >= t_best, a10_te)
cost_example = total_cost(y10_te, p_te10 * a10_te > 30_000, a10_te)
### END CODE HERE ###
print(f"tổng chi phí test (triệu VND): ngưỡng chung = {cost_global:.1f} | ngưỡng phụ thuộc số tiền = {cost_example:.1f} | "
      f"không cảnh báo = {total_cost(y10_te, np.zeros_like(y10_te, bool), a10_te):.1f}")

In [ ]:
check_less(cost_example, cost_global, "quy tắc p·a > chi phí kiểm tra rẻ hơn ngưỡng chung")